# Inspect Power BI TMDL Model and Source Workbook
This notebook loads the existing TMDL table definitions and examines the external Excel workbook used by the model.

In [ ]:
import json
from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET

workspace_root = Path(r'c:/Users/Ahmed Elbadrawy/Downloads/datax_update/update_datax')
model_dir = workspace_root / 'no1.SemanticModel' / 'definition'
workbook_path = Path(r'D:/work/projects/data x projects/New folder/ROW_DATA.xlsx')
print('Model directory:', model_dir)
print('Workbook path:', workbook_path)
print('Tables in model:')
for path in sorted((model_dir / 'tables').glob('*.tmdl')):
    print(' -', path.name)

with zipfile.ZipFile(workbook_path) as z:
    workbook_xml = ET.fromstring(z.read('xl/workbook.xml'))
    ns = {'x': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    sheets = []
    for element in workbook_xml.findall('.//x:sheets/x:sheet', ns):
        rid = element.attrib.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
        sheets.append((element.attrib['name'], rid))
    rels = ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))
    rel_map = {r.attrib['Id']: r.attrib['Target'] for r in rels.findall('{http://schemas.openxmlformats.org/package/2006/relationships}Relationship')}
    print('Workbook sheets and targets:')
    for name, rid in sheets:
        target = rel_map.get(rid)
        print(' -', name, '->', target)
        if target and target.startswith('worksheets/'):
            sheet_xml = ET.fromstring(z.read('xl/' + target))
            cols = []
            shared_strings = None
            if 'xl/sharedStrings.xml' in z.namelist():
                shared_strings = [t.text or '' for si in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('.//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t') for t in [si]]
            for row in sheet_xml.findall('.//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}row'):
                if row.attrib.get('r') == '1':
                    for cell in row.findall('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}c'):
                        value = ''
                        v = cell.find('{http://schemas.openxmlformats.org/spreadsheetml/2006/10/04}v') or cell.find('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}v')
                        if v is not None:
                            value = v.text or ''
                            if cell.attrib.get('t') == 's' and shared_strings is not None:
                                value = shared_strings[int(value)]
                        cols.append(value)
                    print('   headers:', cols)
                    break